# ViBioMIR · Qwen3 Embedding 0.6B · phần 1/3 · T4 ×2

Chọn **GPU T4 ×2**, bật Internet, **Save Version → Save & Run All**.
Notebook dùng **2 tiến trình, 2 bản mô hình**, bắt buộc nhìn thấy hai T4.
Ba phần độc lập, không cần output của phần khác. Dataset input đã được gắn.

Embedding nguyên trường `content`, không thêm prompt tài liệu; last-token pooling,
L2 normalization, 1024 chiều. FP16 trên T4, SDPA; không cài FlashAttention 2.
Tokenize không truncation: nếu vượt 4096 token thì dừng và báo lỗi, không cắt nội dung.

Mỗi file Parquet chứa vector float32, UUID, ID chunk/doc, văn bản và `payload_json`
giữ **toàn bộ record gốc**, kèm `_embedding` để đối chiếu nguồn và mô hình.
Kết quả trong `/kaggle/working/vibiomir_embeddings/part-1/`.
Chỉ dùng bộ output có `manifest.json` ghi `complete: true` để nhập Qdrant.

Dừng có checkpoint sau 11,25 giờ tính từ cell cấu hình, chừa thời gian lưu output.
ETA tính từ tốc độ thực, chưa phải cam kết hoàn thành trong một phiên.
Nếu chưa xong: Add Input output phiên cũ, điền đường dẫn thư mục chứa manifest vào
`CONFIG['resume_roots']`; chạy lại cùng cấu hình. File hoàn tất được kiểm tra checksum
và sao chép vào output mới. Chỉ attach đúng phần tương ứng, tránh input trùng.

Sau khi **cả ba phần** hoàn tất: Add Input các output để sử dụng vector/metadata.
Hướng dẫn tải và tự chạy phần 3 trong `README.md`.


In [ ]:
import time
import json
from pathlib import Path
CONFIG = json.loads('{"part": 1, "model_id": "Qwen/Qwen3-Embedding-0.6B", "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3", "dimension": 1024, "max_length": 4096, "dtype": "float16", "pooling": "last_token", "normalize": true, "document_prompt": "", "vector_name": "qwen3_dense", "output_rows": 8192, "max_batch_size": 64, "padded_token_budget": 16384, "max_hours": 11.25, "expected_source_sha256": "a45123b3de7a9b048f83418515a3ec6b3dd09e1e4491dd361429b787659a3e7f", "input_root": "/kaggle/input", "output_root": "/kaggle/working/vibiomir_embeddings", "resume_roots": [], "require_t4": true}')
CONFIG['deadline_unix'] = time.time() + CONFIG['max_hours'] * 3600
# RESUME: điền thư mục part-N của output cũ đã Add Input.
# CONFIG['resume_roots'] = ['/kaggle/input/.../vibiomir_embeddings/part-1']


In [ ]:
# Giữ PyTorch/CUDA có sẵn của Kaggle, chỉ pin các thư viện cần thiết.
import os
os.environ['HF_HOME'] = '/kaggle/temp/vibiomir-hf'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.57.1', 'huggingface-hub==0.36.0',
                'tokenizers==0.22.1', 'pyarrow==20.0.0', 'zstandard==0.23.0',
                'safetensors==0.6.2'], check=True)


In [ ]:
runtime_path = Path('/kaggle/working/vibiomir_embedding_runtime.py')
runtime_path.write_text('"""Two independent T4 workers, resumable Parquet shards, lossless payloads."""\n\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport time\nimport uuid\n\nimport numpy as np\nimport pyarrow as pa\nimport pyarrow.parquet as pq\nimport zstandard as zstd\n\n\nPOINT_NAMESPACE = uuid.uuid5(uuid.NAMESPACE_URL, "https://vibiomir.org/chunks")\nPROFILE_FIELDS = ("model_id", "revision", "dimension", "max_length", "dtype", "pooling",\n                  "normalize", "document_prompt", "vector_name")\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".partial")\n    with temporary.open("w") as handle:\n        json.dump(value, handle, ensure_ascii=False, indent=2)\n        handle.write("\\n")\n        handle.flush()\n        os.fsync(handle.fileno())\n    temporary.replace(path)\n\n\ndef encoder_profile(config):\n    profile = {key: config[key] for key in PROFILE_FIELDS}\n    profile.update(attention="sdpa", padding_side="left", truncation=False,\n                   output_dtype="float32", point_id="uuid5(vibiomir namespace, chunk_id)")\n    return profile\n\n\ndef profile_id(profile):\n    return hashlib.sha256(json.dumps(profile, sort_keys=True).encode()).hexdigest()\n\n\ndef point_id(chunk_id):\n    return str(uuid.uuid5(POINT_NAMESPACE, chunk_id))\n\n\ndef source_records(path, shard):\n    line_number = shard["start_line"]\n    offset = shard["start_byte"]\n    raw_hash = hashlib.sha256()\n    with Path(path).open("rb") as compressed:\n        with zstd.ZstdDecompressor().stream_reader(compressed) as reader:\n            for line in io.BufferedReader(reader):\n                raw_hash.update(line)\n                yield json.loads(line), line_number, offset\n                line_number += 1\n                offset += len(line)\n    if (line_number - shard["start_line"], offset, raw_hash.hexdigest()) != (\n            shard["rows"], shard["end_byte"], shard["raw_sha256"]):\n        raise ValueError("Decompressed input does not match source manifest")\n\n\ndef batches_by_length(lengths, max_batch_size, padded_token_budget):\n    batch = []\n    longest = 0\n    for index in sorted(range(len(lengths)), key=lengths.__getitem__):\n        next_longest = max(longest, lengths[index])\n        if batch and (len(batch) >= max_batch_size or\n                      next_longest * (len(batch) + 1) > padded_token_budget):\n            yield batch\n            batch, longest = [], 0\n        batch.append(index)\n        longest = max(longest, lengths[index])\n    if batch:\n        yield batch\n\n\ndef embed_records(records, tokenizer, model, config, device):\n    import torch\n    import torch.nn.functional as functional\n\n    features = tokenizer([record[0]["content"] for record in records], padding=False,\n                         truncation=False, add_special_tokens=True)\n    lengths = [len(ids) for ids in features["input_ids"]]\n    if max(lengths) > config["max_length"]:\n        index = lengths.index(max(lengths))\n        raise ValueError(f"Chunk {records[index][0][\'chunk_id\']} has {lengths[index]} tokens; "\n                         "increase max_length explicitly; no content was truncated")\n    vectors = np.empty((len(records), config["dimension"]), dtype=np.float32)\n    queue = list(batches_by_length(lengths, config["max_batch_size"], config["padded_token_budget"]))\n    oom_retries = 0\n    with torch.inference_mode():\n        while queue:\n            indices = queue.pop(0)\n            inputs = hidden = pooled = normalized = None\n            try:\n                inputs = tokenizer.pad([{key: value[index] for key, value in features.items()}\n                                        for index in indices], padding=True, return_tensors="pt")\n                inputs = {key: value.to(device) for key, value in inputs.items()}\n                hidden = model(**inputs, use_cache=False, return_dict=True).last_hidden_state\n                # Left padding means the final position is a real token for every row.\n                pooled = hidden[:, -1].float()\n                normalized = functional.normalize(pooled, p=2, dim=1)\n                vectors[indices] = normalized.cpu().numpy()\n            except torch.OutOfMemoryError:\n                if len(indices) == 1:\n                    raise\n                oom_retries += 1\n                middle = len(indices) // 2\n                queue[0:0] = [indices[:middle], indices[middle:]]\n            finally:\n                del inputs, hidden, pooled, normalized\n            if oom_retries:\n                torch.cuda.empty_cache()\n    norms = np.linalg.norm(vectors, axis=1)\n    if not np.isfinite(vectors).all() or not np.allclose(norms, 1, atol=2e-5):\n        raise ValueError("Invalid embedding vectors")\n    return vectors, lengths, oom_retries\n\n\ndef parquet_schema(dimension, metadata):\n    return pa.schema([\n        ("point_id", pa.string()), ("chunk_id", pa.string()), ("doc_id", pa.int64()),\n        ("source_line", pa.int64()), ("source_byte", pa.int64()),\n        ("content", pa.large_string()), ("content_origin", pa.string()),\n        ("token_count", pa.int32()), ("input_token_count", pa.int32()),\n        ("content_sha256", pa.string()), ("payload_json", pa.large_string()),\n        ("vector", pa.list_(pa.float32(), dimension)),\n    ], metadata={b"vibiomir_embedding": json.dumps(metadata, sort_keys=True).encode()})\n\n\ndef write_shard(path, records, vectors, lengths, config, source_manifest, gpu):\n    profile = encoder_profile(config)\n    metadata = dict(schema_version="vibiomir-embedding-v1", encoder=profile,\n                    profile_id=profile_id(profile), source_sha256=source_manifest["source_sha256"],\n                    part=config["part"], gpu=gpu)\n    columns = {name: [] for name in parquet_schema(config["dimension"], metadata).names if name != "vector"}\n    for (record, line_number, offset), length in zip(records, lengths):\n        if "_embedding" in record:\n            raise ValueError("Input already contains reserved _embedding metadata")\n        content_hash = hashlib.sha256(record["content"].encode()).hexdigest()\n        payload = dict(record)\n        payload["_embedding"] = dict(source_file=source_manifest["source_file"],\n            source_sha256=source_manifest["source_sha256"], source_line=line_number,\n            source_byte=offset, content_sha256=content_hash, profile_id=metadata["profile_id"],\n            model_id=config["model_id"], revision=config["revision"], part=config["part"],\n            gpu=gpu, input_token_count=length, truncated=False)\n        values = dict(point_id=point_id(record["chunk_id"]), chunk_id=record["chunk_id"],\n                      doc_id=record["doc_id"], source_line=line_number, source_byte=offset,\n                      content=record["content"], content_origin=record.get("content_origin"),\n                      token_count=record.get("token_count"), input_token_count=length,\n                      content_sha256=content_hash, payload_json=json.dumps(payload, ensure_ascii=False))\n        for key, value in values.items():\n            columns[key].append(value)\n    schema = parquet_schema(config["dimension"], metadata)\n    arrays = [pa.array(columns[field.name], type=field.type) for field in schema if field.name != "vector"]\n    arrays.append(pa.FixedSizeListArray.from_arrays(pa.array(vectors.ravel()), config["dimension"]))\n    table = pa.Table.from_arrays(arrays, schema=schema)\n    temporary = Path(str(path) + ".partial")\n    pq.write_table(table, temporary, compression="zstd", compression_level=3,\n                   row_group_size=1024, use_dictionary=["content_origin"])\n    temporary.replace(path)\n    receipt = dict(file=path.name, rows=len(records), sha256=file_hash(path),\n                   start_line=records[0][1], end_line=records[-1][1],\n                   start_byte=records[0][2], part=config["part"], gpu=gpu,\n                   source_sha256=source_manifest["source_sha256"], profile_id=metadata["profile_id"],\n                   min_tokens=min(lengths), max_tokens=max(lengths))\n    atomic_json(Path(str(path) + ".json"), receipt)\n    return receipt\n\n\ndef validate_shard(path, receipt, config, source_manifest):\n    if receipt["profile_id"] != profile_id(encoder_profile(config)):\n        raise ValueError("Checkpoint encoder configuration differs")\n    if receipt["source_sha256"] != source_manifest["source_sha256"]:\n        raise ValueError("Checkpoint corpus differs")\n    if file_hash(path) != receipt["sha256"]:\n        raise ValueError(f"Checkpoint checksum mismatch: {path}")\n    parquet = pq.ParquetFile(path)\n    if parquet.metadata.num_rows != receipt["rows"]:\n        raise ValueError("Checkpoint row count differs")\n    if receipt["rows"] != receipt["end_line"] - receipt["start_line"] + 1:\n        raise ValueError("Checkpoint has inconsistent source range")\n    metadata = json.loads(parquet.schema_arrow.metadata[b"vibiomir_embedding"])\n    if metadata["profile_id"] != receipt["profile_id"] or metadata["source_sha256"] != receipt["source_sha256"]:\n        raise ValueError("Parquet encoder/source metadata differs")\n    next_line = receipt["start_line"]\n    for batch in parquet.iter_batches(batch_size=1024, columns=["source_line", "vector"]):\n        lines = batch.column(0).to_numpy()\n        if not np.array_equal(lines, np.arange(next_line, next_line + len(lines))):\n            raise ValueError("Checkpoint has missing or reordered rows")\n        next_line += len(lines)\n        values = batch.column(1).values.to_numpy().reshape(len(lines), config["dimension"])\n        if not np.isfinite(values).all() or not np.allclose(np.linalg.norm(values, axis=1), 1, atol=2e-5):\n            raise ValueError("Checkpoint contains invalid vectors")\n\n\ndef restore_shards(output, shard, config, source_manifest, gpu):\n    candidates = {}\n    for root in [output, *map(Path, config["resume_roots"])]:\n        if not root.is_dir():\n            raise ValueError(f"Resume directory does not exist: {root}")\n        for path in root.glob(f"p{config[\'part\']}-gpu{gpu}-*.parquet.json"):\n            receipt = json.loads(path.read_text())\n            if Path(receipt["file"]).name != receipt["file"]:\n                raise ValueError("Invalid checkpoint filename")\n            key = receipt["start_line"]\n            if key in candidates and candidates[key][1] != receipt:\n                raise ValueError("Conflicting checkpoints for the same source range")\n            candidates[key] = (path.parent / receipt["file"], receipt)\n    receipts = []\n    next_line = shard["start_line"]\n    for source, receipt in sorted(candidates.values(), key=lambda item: item[1]["start_line"]):\n        if receipt["start_line"] != next_line or receipt["end_line"] > shard["end_line"]:\n            raise ValueError("Checkpoint is not a contiguous prefix of this GPU\'s source range")\n        if receipt["part"] != config["part"] or receipt["gpu"] != gpu:\n            raise ValueError("Checkpoint belongs to another worker")\n        validate_shard(source, receipt, config, source_manifest)\n        destination = output / receipt["file"]\n        if source.resolve() != destination.resolve():\n            temporary = Path(str(destination) + ".partial")\n            shutil.copyfile(source, temporary)\n            temporary.replace(destination)\n            atomic_json(Path(str(destination) + ".json"), receipt)\n        receipts.append(receipt)\n        next_line = receipt["end_line"] + 1\n    return receipts\n\n\ndef run(config):\n    import torch\n    from huggingface_hub import snapshot_download\n    from transformers import AutoModel, AutoTokenizer\n\n    started = time.time()\n    rank = int(os.environ["LOCAL_RANK"])\n    if int(os.environ.get("WORLD_SIZE", "0")) != 2 or torch.cuda.device_count() != 2:\n        raise RuntimeError("Exactly two visible GPUs and two torchrun workers are required")\n    device = f"cuda:{rank}"\n    torch.cuda.set_device(rank)\n    name = torch.cuda.get_device_name(rank)\n    if config["require_t4"] and "T4" not in name:\n        raise RuntimeError(f"GPU {rank} is {name}; select Kaggle GPU T4 x2")\n    torch.set_num_threads(2)\n    print(f"[GPU {rank}] {name}; loading a dedicated model replica", flush=True)\n    inputs = Path(config["input_root"])\n    manifests = list(inputs.rglob("source_manifest.json"))\n    matches = [(path, json.loads(path.read_text())) for path in manifests]\n    matches = [(path, data) for path, data in matches\n               if data.get("source_sha256") == config["expected_source_sha256"]]\n    if len(matches) != 1:\n        raise ValueError(f"Expected exactly one matching source manifest, found {len(matches)}")\n    manifest_path, source_manifest = matches[0]\n    shard = next(s for s in source_manifest["shards"] if s["part"] == config["part"] and s["gpu"] == rank)\n    source = manifest_path.parent / shard["file"]\n    if file_hash(source) != shard["compressed_sha256"]:\n        raise ValueError("Compressed input checksum mismatch")\n    output = Path(config["output_root"]) / f"part-{config[\'part\']}"\n    output.mkdir(parents=True, exist_ok=True)\n    receipts = restore_shards(output, shard, config, source_manifest, rank)\n    resumed = sum(item["rows"] for item in receipts)\n    # Both workers share HF cache; Hub file locks protect the download.\n    model_path = snapshot_download(config["model_id"], revision=config["revision"],\n                                   allow_patterns=["*.json", "*.txt", "*.safetensors"], max_workers=2)\n    tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left")\n    model = AutoModel.from_pretrained(model_path, torch_dtype=torch.float16,\n                                     attn_implementation="sdpa").to(device).eval()\n    if model.config.hidden_size != config["dimension"]:\n        raise ValueError("Unexpected model embedding dimension")\n    pending = []\n    origins = Counter()\n    new_rows = 0\n    total_oom = 0\n    inference_start = time.time()\n\n    def commit(records):\n        nonlocal new_rows, total_oom\n        vectors, lengths, oom = embed_records(records, tokenizer, model, config, device)\n        path = output / f"p{config[\'part\']}-gpu{rank}-{records[0][1]:09d}.parquet"\n        receipt = write_shard(path, records, vectors, lengths, config, source_manifest, rank)\n        receipts.append(receipt)\n        new_rows += len(records)\n        total_oom += oom\n        origins.update(row[0].get("content_origin", "unknown") for row in records)\n        seconds = time.time() - inference_start\n        rate = new_rows / max(seconds, 1)\n        progress = dict(gpu=rank, device=name, rows=resumed + new_rows, expected_rows=shard["rows"],\n                        rows_per_second=round(rate, 2),\n                        eta_hours=round((shard["rows"] - resumed - new_rows) / max(rate, .001) / 3600, 2),\n                        max_allocated_gib=round(torch.cuda.max_memory_allocated(rank) / 2**30, 3))\n        atomic_json(output / f"progress-gpu{rank}.json", progress)\n        print(json.dumps(progress), flush=True)\n\n    for record in source_records(source, shard):\n        if record[1] < shard["start_line"] + resumed:\n            continue\n        if time.time() >= config["deadline_unix"]:\n            break\n        pending.append(record)\n        if len(pending) == config["output_rows"]:\n            commit(pending)\n            pending = []\n    if pending:\n        commit(pending)\n    count = sum(item["rows"] for item in receipts)\n    worker = dict(run_id=config["run_id"], gpu=rank, device=name, rows=count, expected_rows=shard["rows"],\n                  resumed_rows=resumed, new_rows=new_rows, complete=count == shard["rows"],\n                  elapsed_seconds=round(time.time() - started, 2), oom_batch_splits=total_oom,\n                  new_content_origins=dict(origins), files=receipts,\n                  peak_allocated_gib=round(torch.cuda.max_memory_allocated(rank) / 2**30, 3))\n    atomic_json(output / f"worker-gpu{rank}.json", worker)\n    # Release GPU memory before waiting for the other worker\'s final receipt.\n    del model\n    torch.cuda.empty_cache()\n    if rank == 0:\n        other = output / "worker-gpu1.json"\n        while True:\n            other_worker = json.loads(other.read_text()) if other.exists() else {}\n            if other_worker.get("run_id") == config["run_id"]:\n                break\n            time.sleep(2)\n        workers = [worker, other_worker]\n        if any(w["rows"] == 0 for w in workers):\n            raise RuntimeError("Both GPUs must process nonempty source partitions")\n        summary = dict(schema_version="vibiomir-embedding-v1", part=config["part"],\n                       complete=all(w["complete"] for w in workers),\n                       rows=sum(w["rows"] for w in workers),\n                       expected_rows=sum(w["expected_rows"] for w in workers),\n                       source=source_manifest, encoder=encoder_profile(config),\n                       profile_id=profile_id(encoder_profile(config)), gpu_workers=workers,\n                       files=[entry for w in workers for entry in w["files"]],\n                       elapsed_seconds=round(time.time() - started, 2))\n        atomic_json(output / "manifest.json", summary)\n        print(json.dumps(dict(part=config["part"], complete=summary["complete"],\n                              rows=summary["rows"], expected_rows=summary["expected_rows"])), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    arguments = parser.parse_args()\n    run(json.loads(Path(arguments.config).read_text()))\n')
output = Path(CONFIG['output_root']) / f"part-{CONFIG['part']}"
output.mkdir(parents=True, exist_ok=True)
(output / 'README.md').write_text("# ViBioMIR — 3 phiên Qwen3 Embedding 0.6B, mỗi phiên T4 ×2\n\nToàn bộ **4.936.835 chunk** được chia thành **3 phần độc lập**, cân bằng theo tổng\n`token_count` gốc. Mỗi phần gồm hai dải dòng liên tục, mỗi GPU xử lý một dải.\nPhần 1 và 2 được đưa lên Kaggle. Phần 3 có notebook và gói input riêng để tự chạy.\nKhông cần output của phần trước để chạy phần sau. Chưa tạo/nạp Qdrant ở bước này.\n\n## Cấu hình và thời gian\n\n- Settings: **GPU T4 ×2**, **Internet On**. Không chọn P100 hoặc một GPU.\n- `torchrun --nproc_per_node=2`: hai tiến trình, mỗi GPU giữ một model replica.\n  Runtime bắt buộc nhìn thấy đúng hai T4, không tự rơi về một GPU/CPU.\n- Model `Qwen/Qwen3-Embedding-0.6B`, revision cố định\n  `97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3`.\n- FP16, SDPA, left padding, last-token pooling, L2 normalization, vector 1024 chiều.\n  Giữ PyTorch/CUDA có sẵn của Kaggle; notebook cài các thư viện Python cần thiết.\n  Không cài FlashAttention 2 trên T4.\n- Chỉ embedding nguyên trường `content`, không thêm prompt/chat template.\n  Tokenize không truncation; nếu vượt 4096 token thì báo lỗi, không cắt văn bản.\n- Sắp theo độ dài trong block, tối đa 64 chunk/batch và 16.384 padded token.\n  Batch OOM được chia đôi. Khi ghi output, trả về đúng thứ tự nguồn.\n- Tối đa 8.192 dòng/file Parquet, nén Zstandard. Không gom mọi vector vào RAM.\n- Dừng nhận block mới sau **11,25 giờ** từ cell cấu hình, hoàn tất block đang chạy\n  và lưu checkpoint, chừa thời gian xuất output trước giới hạn 12 giờ.\n- Ghi tốc độ/ETA thực tế riêng mỗi GPU. Chưa đo trên Kaggle thì chưa thể cam kết\n  hoàn tất trong một phiên; nếu chưa xong, tiếp tục từ checkpoint như bên dưới.\n\n## Tải và setup phần 3 trên Kaggle\n\nGói `ViBioMIR_Part3_Kaggle.zip` chứa:\n\n```text\nViBioMIR_Qwen3_06B_Part3_T4x2.ipynb\nREADME.md\ninput/\n  chunks-04.jsonl.zst\n  chunks-05.jsonl.zst\n  source_manifest.json\n  dataset-metadata.json\n```\n\n1. Tải gói ZIP về máy và giải nén. Giữ nguyên hai file `.jsonl.zst`; **không cần\n   giải nén JSONL**. Chỉ cần hai shard này cho phần 3, không cần phần 1 và 2.\n2. Trong Kaggle, vào **Datasets → New Dataset**. Upload hai file `.jsonl.zst` và\n   `source_manifest.json` trong thư mục `input/`, chọn **Private**, rồi Create.\n   Không upload notebook hoặc ZIP bao ngoài làm dataset input. `dataset-metadata.json`\n   chỉ cần nếu dùng CLI, không bắt buộc với giao diện web.\n3. Vào **Code → New Notebook → File → Import Notebook**, chọn\n   `ViBioMIR_Qwen3_06B_Part3_T4x2.ipynb` đã giải nén.\n4. **Add Input** dataset phần 3 vừa tạo. Nếu dùng cùng tài khoản `nhminh107`, có thể\n   gắn dataset nguồn 3 phần đã upload sẵn thay cho việc tạo dataset mới.\n   Chỉ attach một dataset có `source_manifest.json` tương ứng để tránh bị trùng.\n5. Settings: **Accelerator → GPU T4 ×2**, **Internet → On**.\n6. Giữ `CONFIG['part'] = 3`, rồi **Save Version → Save & Run All**.\n7. Trong log phải thấy cả `[GPU 0] Tesla T4` và `[GPU 1] Tesla T4`, sau đó có tiến độ\n   `gpu: 0` và `gpu: 1`. Nếu thiếu GPU, runtime báo lỗi trước khi tải mô hình.\n8. Khi hoàn tất, kiểm tra `manifest.json`: **`complete: true`** và\n   **`rows == expected_rows`**. Lưu output version này để Add Input về sau.\n\nInput giữ nguyên record nguồn theo byte. Manifest liệt kê sáu shard là bình thường:\nnotebook phần 3 chỉ đọc shard 04 và 05. Tính duy nhất của toàn bộ `chunk_id` đã được\nkiểm tra bằng SQLite trên đĩa, cùng SHA-256 của toàn bộ JSONL gốc.\n\n## Output và metadata\n\nTrong `/kaggle/working/vibiomir_embeddings/part-N/`:\n\n- `pN-gpuG-XXXXXXXXX.parquet`: vector và metadata từng block.\n- `*.parquet.json`: receipt ghi SHA-256, số dòng, dải dòng nguồn và profile.\n- `progress-gpu0.json`, `progress-gpu1.json`: số dòng, tốc độ, ETA và bộ nhớ GPU.\n- `worker-gpu*.json`: số chunk thực sự xử lý/khôi phục trên mỗi GPU.\n- `manifest.json`: trạng thái hoàn thành, danh sách file, source và encoder profile.\n- `README.md`: hướng dẫn này.\n\n| Cột Parquet | Ý nghĩa |\n| --- | --- |\n| `point_id` | UUID5 từ namespace cố định và `chunk_id`; chạy lại giữ ID |\n| `chunk_id`, `doc_id` | ID nguồn giữ nguyên kiểu string/int64 |\n| `source_line`, `source_byte` | Dòng **1-based**, byte **0-based** trong JSONL gốc chưa nén |\n| `content`, `content_origin`, `token_count` | Văn bản nguyên gốc, nguồn recovery, số token gốc |\n| `input_token_count` | Số token thực sự đưa vào Qwen |\n| `content_sha256` | SHA-256 UTF-8 của văn bản chính xác đã embedding |\n| `payload_json` | Toàn bộ record gốc và trường bổ sung `_embedding` |\n| `vector` | Fixed-size list float32, 1024 chiều, chuẩn L2 bằng 1 |\n\n`payload_json` giữ đầy đủ mọi trường metadata gốc. `_embedding` gồm model/revision,\nprofile ID, hash nguồn/nội dung, số dòng/byte, phần, GPU, độ dài input và\n`truncated=false`. Metadata schema của Parquet cũng lưu đầy đủ encoder profile.\nKhông bỏ chunk recovery, không đổi ID, không làm tròn vector sang FP16 khi lưu.\n\nMỗi phần có khoảng 6,7 GB vector float32 thô cộng payload; Parquet nén giúp giảm\nvăn bản. Không tạo ZIP thứ hai của output trong working.\n\n## Tiếp tục nếu chưa hoàn tất\n\n1. Lưu output phiên đã dừng. Tạo version mới của **cùng notebook/phần**.\n2. Add Input output phiên cũ hoặc dataset chứa nguyên thư mục output.\n3. Trong cell cấu hình, đặt `CONFIG['resume_roots']` thành danh sách thư mục `part-N`\n   chứa các file `*.parquet.json`, ví dụ\n   `['/kaggle/input/your-previous-output/vibiomir_embeddings/part-3']`.\n4. Giữ nguyên nguồn và cấu hình encoder. Run All.\n\nRuntime kiểm tra checksum, dimension, vector, thứ tự dòng và fingerprint trước khi\nkhôi phục, rồi sao chép block hoàn tất vào output mới để version mới tự đủ dữ liệu.\nFile `.partial` và Parquet chưa có receipt không được coi là checkpoint hoàn tất.\nTrong cùng phiên bị gián đoạn, có thể chạy lại cell launcher để tiếp tục file đã ghi.\n\n## Add Input để dùng vector về sau\n\nKhi cả ba manifest đều có `complete: true`, Add Input ba output vào notebook xử lý\nbước tiếp theo. Nếu chạy bằng tài khoản Kaggle khác, tải output phần 3 về rồi upload\nthành dataset private trên tài khoản đích, hoặc cấp quyền truy cập phù hợp.\n\nVí dụ đọc theo batch, không nạp toàn bộ dữ liệu vào RAM:\n\n```python\nfrom pathlib import Path\nimport json\nimport pyarrow.parquet as pq\n\npart = Path('/kaggle/input/YOUR-OUTPUT/vibiomir_embeddings/part-3')\nmanifest = json.loads((part / 'manifest.json').read_text())\nassert manifest['complete'] and manifest['rows'] == manifest['expected_rows']\n\nfor item in manifest['files']:\n    for batch in pq.ParquetFile(part / item['file']).iter_batches(batch_size=128):\n        rows = batch.to_pydict()\n        # rows['vector'], rows['point_id'], rows['chunk_id'], rows['doc_id']\n        # metadata đầy đủ: json.loads(rows['payload_json'][i])\n```\n\nGiữ đủ manifest/receipt/Parquet để kiểm tra checksum và ba phần cùng profile.\nĐầu ra có ID hợp lệ và payload đầy đủ để dùng với hệ lưu trữ ở bước sau.\n\n## Tạo lại gói local\n\n```sh\n.venv/bin/python Scripts/Kaggle/prepare_vibiomir_embeddings.py\n# Sau khi sửa runtime/tài liệu, không chia lại dữ liệu:\n.venv/bin/python Scripts/Kaggle/prepare_vibiomir_embeddings.py --notebook-only\n```\n\nGói upload: `artifacts/vibiomir-qwen3-3parts-20261024/`.\nChỉ publish/run phần 1 và 2; phần 3 do người dùng tự chạy. Mặc định private,\naccelerator `NvidiaTeslaT4`, timeout 43.200 giây.\n\nNguồn kỹ thuật: [Qwen model card](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B),\n[Kaggle metadata](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels_metadata.md).\n")
config_path = Path('/kaggle/working/vibiomir_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2))


In [ ]:
# torchrun tạo đúng 2 worker; runtime kiểm tra tên GPU và số chunk từng GPU.
import uuid
CONFIG['run_id'] = str(uuid.uuid4())
config_path.write_text(json.dumps(CONFIG, indent=2))
subprocess.run([sys.executable, '-m', 'torch.distributed.run',
                '--standalone', '--nproc_per_node=2', str(runtime_path),
                '--config', str(config_path)], check=True)


In [ ]:
report = json.loads((output / 'manifest.json').read_text())
print(json.dumps({k: report[k] for k in ('part', 'complete', 'rows', 'expected_rows',
                                       'elapsed_seconds')}, indent=2))
print(json.dumps([{k: worker[k] for k in ('gpu', 'device', 'rows', 'new_rows',
                                          'resumed_rows', 'peak_allocated_gib')}
                  for worker in report['gpu_workers']], indent=2))
if report['complete']:
    print('HOÀN TẤT phần', CONFIG['part'], '— có thể Add Input output này để nạp Qdrant.')
else:
    print('CHƯA HOÀN TẤT. Đã lưu checkpoint. Add Input output này và đặt resume_roots để tiếp tục.')
